In [1]:
import threading
from flask import Flask, request, jsonify, redirect, session
from breeze_connect import BreezeConnect
import urllib.parse

app = Flask(__name__)
app.secret_key = "1997"

API_KEY = "4_8463B9784119l72yC4426912px346P"
API_SECRET = "816O1S=eG215716g518CM)X225Z0!78b"
ACCESS_TOKEN = None

breeze = BreezeConnect(api_key=API_KEY)

@app.route('/')
def login():
    login_url = f"https://api.icicidirect.com/apiuser/login?api_key={urllib.parse.quote_plus(API_KEY)}"
    return redirect(login_url)

@app.route('/callback', methods=['POST'])
def callback():
    global ACCESS_TOKEN

    session_token = request.args.get("apisession")
    if not session_token:
        return "Authorization failed", 400

    try:
        breeze.generate_session(api_secret=API_SECRET, session_token=session_token)
        ACCESS_TOKEN = session_token
        session["access_token"] = ACCESS_TOKEN
        
        # Fetch customer details
        customer_details = breeze.get_customer_details(api_session=ACCESS_TOKEN)
        return jsonify({"message": "Session active", "customer_details": customer_details})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Start Flask in a separate thread
def run_flask():
    app.run(debug=True, use_reloader=False)

threading.Thread(target=run_flask).start()

 * Serving Flask app '__main__'


 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [19/Apr/2025 13:09:32] "GET / HTTP/1.1" 302 -
127.0.0.1 - - [19/Apr/2025 13:10:15] "POST /callback?apisession=51234978 HTTP/1.1" 200 -
127.0.0.1 - - [19/Apr/2025 13:10:15] "GET /favicon.ico HTTP/1.1" 404 -


In [17]:
import os
from datetime import datetime
import pandas as pd

# Function to format date and datetime in ISO format
def format_iso(date_str, time_str="07:00:00"):
    return f"{datetime.strptime(date_str, '%d/%m/%Y').date()}T{time_str}.000Z"

# Define parameters
start_date = format_iso("01/01/2025", "09:15:00")
end_date = format_iso("01/01/2025", "15:30:00")
exp_date = format_iso("02/01/2025", "15:30:00")

stock_code = "NIFTY"
exchange_code = "NFO"
interval = "1minute"
product_type = "options"
rights = ["call", "put"]

central_strike = 23650

# Generate list of 5 strikes below, central, and 5 above
strike_prices = [str(central_strike + i * 50) for i in range(-7, 7)]

filename = "02-Jan-25.xlsx"
all_data = []

# Loop through each strike and right (call/put)
for strike_price in strike_prices:
    for right in rights:
        data = breeze.get_historical_data(
            interval=interval,
            from_date=start_date,
            to_date=end_date,
            expiry_date=exp_date,
            stock_code=stock_code,
            strike_price=strike_price,
            product_type=product_type,
            exchange_code=exchange_code,
            right=right
        )

        if "Success" in data and isinstance(data["Success"], list):
            df = pd.DataFrame(data["Success"])
            all_data.append(df)

# Combine and save if data was fetched
if all_data:
    new_data = pd.concat(all_data, ignore_index=True)

    if os.path.exists(filename):
        existing_data = pd.read_excel(filename, engine="openpyxl")
        combined_data = pd.concat([existing_data, new_data], ignore_index=True)
        # combined_data.drop_duplicates(inplace=True)
    else:
        combined_data = new_data

    if 'datetime' in combined_data.columns:
        combined_data['datetime'] = pd.to_datetime(combined_data['datetime'])
        combined_data['date'] = combined_data['datetime'].dt.date
        combined_data['time'] = combined_data['datetime'].dt.time

    combined_data.to_excel(filename, index=False, engine="openpyxl")
    print(f"Data successfully updated in {filename}")
else:
    print("No new valid data received.")


Data successfully updated in 02-Jan-25.xlsx


In [18]:
import pandas as pd

# Load both Excel files
df1 = pd.read_excel("02-Jan-25.xlsx", engine="openpyxl")
df2 = pd.read_excel("09-Jan-25.xlsx", engine="openpyxl")
df3 = pd.read_excel("16-Jan-25.xlsx", engine="openpyxl")
df4 = pd.read_excel("23-Jan-25.xlsx", engine="openpyxl")
df5 = pd.read_excel("30-Jan-25.xlsx", engine="openpyxl")

# Combine them
combined_df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

# Save the result
combined_df.to_excel("Jan2025.xlsx", index=False, engine="openpyxl")
print("Files combined and saved as 'combined_file.xlsx'")

Files combined and saved as 'combined_file.xlsx'
